# 02 Small-Batch Extraction Probe

**Project**: AIGC Research Comprehension System  
**Purpose**: Run a rule-based extraction probe over a sample of 8 papers to validate data quality and the feasibility of automated claim extraction.

**Constraints**:
- No LLMs, embeddings, or training.
- Rule-based heuristics only.
- Reads from persistent Google Drive storage.

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


## 2. Define Paths and Sample IDs

In [2]:
from pathlib import Path
import pandas as pd
import json
from collections import Counter, defaultdict
from datetime import datetime
import re
import os

BASE = Path("/content/drive/MyDrive/AIGC")
DATA_DIR = BASE / "Data"
REGISTRY_DIR = DATA_DIR / "registry"
PARSED_DIR = DATA_DIR / "parsed"
SECTIONS_DIR = DATA_DIR / "sections"
TABLES_DIR = DATA_DIR / "tables"
PROBES_DIR = DATA_DIR / "probes"
PROBES_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_PAPER_IDS = ["P001", "P002", "P003", "P004", "P005", "P007", "P009", "P010"]

## 3. Verify Inputs

In [3]:
required_files = [
    REGISTRY_DIR / "manifest_100.csv",
    REGISTRY_DIR / "document_registry.csv",
    REGISTRY_DIR / "parse_registry.csv",
    SECTIONS_DIR / "sections.jsonl",
    TABLES_DIR / "table_candidates.jsonl"
]

for f in required_files:
    assert f.exists(), f"Missing required input: {f}"

parsed_json_count = len(list(PARSED_DIR.glob("*.json")))
print(f"Parsed JSON files: {parsed_json_count}")

with open(SECTIONS_DIR / "sections.jsonl", "r") as f:
    section_count = sum(1 for _ in f)
print(f"Section rows: {section_count}")

with open(TABLES_DIR / "table_candidates.jsonl", "r") as f:
    table_rows = sum(1 for _ in f)
print(f"Table candidate rows: {table_rows}")

Parsed JSON files: 72
Section rows: 733
Table candidate rows: 7432


## 4. Load Data (Filtered)

In [4]:
manifest = pd.read_csv(REGISTRY_DIR / "manifest_100.csv")
doc_registry = pd.read_csv(REGISTRY_DIR / "document_registry.csv")
parse_registry = pd.read_csv(REGISTRY_DIR / "parse_registry.csv")

sample_sections = []
with open(SECTIONS_DIR / "sections.jsonl", "r") as f:
    for line in f:
        item = json.loads(line)
        if item["paper_id"] in SAMPLE_PAPER_IDS:
            sample_sections.append(item)

sample_tables = []
with open(TABLES_DIR / "table_candidates.jsonl", "r") as f:
    for line in f:
        item = json.loads(line)
        if item["paper_id"] in SAMPLE_PAPER_IDS:
            sample_tables.append(item)

print(f"Loaded {len(sample_sections)} sections for {len(SAMPLE_PAPER_IDS)} papers.")
print(f"Loaded {len(sample_tables)} table candidates.")

Loaded 73 sections for 8 papers.
Loaded 842 table candidates.


## 5. Section Quality Audit

In [5]:
report_lines = ["# Section Quality Sample Report", ""]
paper_metrics = defaultdict(dict)

for pid in SAMPLE_PAPER_IDS:
    sections = [s for s in sample_sections if s["paper_id"] == pid]
    types = [s["section_type"] for s in sections]
    type_counts = Counter(types)

    has_abs_intro = "abstract" in types or "introduction" in types
    has_method = "method" in types
    has_results = "results" in types or "experiment" in types

    issues = []
    if not has_abs_intro: issues.append("no_abstract_or_intro")
    if not has_method: issues.append("no_method")
    if not has_results: issues.append("no_results")
    if type_counts.get("unknown", 0) > 5: issues.append("too_many_unknowns")

    total_len = sum(len(s["text"]) for s in sections)
    if total_len < 5000: issues.append("very_short_text")

    report_lines.append(f"## {pid}")
    report_lines.append(f"- Sections found: {len(sections)}")
    report_lines.append(f"- Types: {dict(type_counts)}")
    report_lines.append(f"- Total Chars: {total_len}")
    if issues: report_lines.append(f"- **Issues**: {', '.join(issues)}")

    # Sample snippets
    for stype in ["abstract", "introduction", "method", "results"]:
        sect = next((s for s in sections if s["section_type"] == stype), None)
        if sect:
            report_lines.append(f"- **{stype.capitalize()} Snippet**: {sect['text'][:300]}...")
    report_lines.append("")

output_file = PROBES_DIR / "section_quality_sample.md"
with open(output_file, "w") as f:
    f.write("\n".join(report_lines))
print(f"Section quality report saved: {output_file}")

Section quality report saved: /content/drive/MyDrive/AIGC/Data/probes/section_quality_sample.md


## 6. Rule-Based Entity Extraction

In [6]:
ENTITIES = {
    "datasets": [
        "GenImage", "Synthbuster", "ForenSynths", "CIFAKE", "FaceForensics++",
        "Celeb-DF", "ImageNet", "COCO", "LAION", "DiffusionDB", "MS-COCO",
        "ArtiFact", "AIGIBench", "WildFake", "DeepFakeDetection", "UniversalFakeDetect"
    ],
    "models": [
        "ResNet", "EfficientNet", "ViT", "CLIP", "DINO", "DINOv2", "DINOv3",
        "CNN", "Xception", "Swin", "ConvNeXt", "SigLIP", "EVA", "MAE", "Vision Transformer"
    ],
    "generators": [
        "GAN", "diffusion", "latent diffusion", "Stable Diffusion", "Midjourney",
        "DALL-E", "Flux", "SDXL", "autoregressive"
    ],
    "metrics": [
        "accuracy", "acc", "AUC", "AP", "F1", "EER", "precision", "recall", "ROC-AUC"
    ],
    "distortions": [
        "JPEG", "blur", "resize", "compression", "WebP", "Gaussian noise", "crop", "downsampling"
    ]
}

if "paper_id" in manifest.columns:
    titles = dict(zip(manifest["paper_id"], manifest["title"]))
elif "paper_id" in doc_registry.columns and "title" in doc_registry.columns:
    titles = dict(zip(doc_registry["paper_id"], doc_registry["title"]))
else:
    titles = {}

entity_results = []
for sect in sample_sections:
    text = sect["text"]
    for etype, patterns in ENTITIES.items():
        for p in patterns:
            if re.search(r"\b" + re.escape(p) + r"\b", text, re.IGNORECASE):
                match = re.search(r"\b" + re.escape(p) + r"\b", text, re.IGNORECASE)
                start = max(0, match.start() - 50)
                end = min(len(text), match.end() + 50)
                entity_results.append({
                    "paper_id": sect["paper_id"],
                    "paper_title": titles.get(sect["paper_id"], "Unknown"),
                    "entity_type": etype,
                    "entity": p,
                    "section_type": sect["section_type"],
                    "section_heading": sect["section_heading"],
                    "evidence_anchor": sect["evidence_anchor"],
                    "context_snippet": text[start:end].replace("\n", " ")
                })

output_file = PROBES_DIR / "entity_probe_sample.csv"
df_entities = pd.DataFrame(entity_results)
df_entities.to_csv(output_file, index=False)
print(f"Extracted {len(entity_results)} entity occurrences. Saved: {output_file}")

if len(entity_results) > 0:
    from IPython.display import display
    display(df_entities.head(10))

Extracted 417 entity occurrences. Saved: /content/drive/MyDrive/AIGC/Data/probes/entity_probe_sample.csv


,paper_id,paper_title,entity_type,entity,section_type,section_heading,evidence_anchor,context_snippet
0,P001,Detecting GAN generated Fake Images using Co-o...,models,CNN,unknown,None,P001:p1-p3:unknown,model using a deep convolutional neural netwo...
1,P001,Detecting GAN generated Fake Images using Co-o...,generators,GAN,unknown,None,P001:p1-p3:unknown,lume: 31 | Article ID: art00008 [image] Detect...
2,P001,Detecting GAN generated Fake Images using Co-o...,metrics,accuracy,unknown,None,P001:p1-p3:unknown,omising and achieves more than 99% classificat...
3,P002,Synthbuster: Towards Detection of Diffusion Mo...,datasets,Synthbuster,unknown,None,P002:p1-p8:unknown,gital Object Identifier 10.1109/XXXX.2022.1234...
4,P002,Synthbuster: Towards Detection of Diffusion Mo...,datasets,ArtiFact,unknown,None,P002:p1-p8:unknown,"2, 4, and 8. Firefly even features a 16-perio..."
5,P002,Synthbuster: Towards Detection of Diffusion Mo...,models,CLIP,unknown,None,P002:p1-p8:unknown,"omputer vision, primarily fueled by the advent..."
6,P002,Synthbuster: Towards Detection of Diffusion Mo...,generators,GAN,unknown,None,P002:p1-p8:unknown,"ng. It has been noted [21], [25], [38], [55] t..."
7,P002,Synthbuster: Towards Detection of Diffusion Mo...,generators,diffusion,unknown,None,P002:p1-p8:unknown,XX.2022.1234567 Synthbuster: Towards Detection...
8,P002,Synthbuster: Towards Detection of Diffusion Mo...,generators,latent diffusion,unknown,None,P002:p1-p8:unknown,a shared space. Capitalizing on this capabili...
9,P002,Synthbuster: Towards Detection of Diffusion Mo...,generators,Stable Diffusion,unknown,None,P002:p1-p8:unknown,o(s) and publication title will appear here.> ...


## 7. Result/Table Tuple Extraction

In [8]:
# --- 7. Result/Table Tuple Extraction ---

result_results = []

def get_first_available(row, keys, default=""):
    """Safely get the first available key from a dict-like row."""
    for k in keys:
        if k in row and row[k] is not None:
            return row[k]
    return default

def find_entity(text, entity_list):
    for ent in entity_list:
        pattern = r"\b" + re.escape(ent) + r"\b"
        if re.search(pattern, text, re.IGNORECASE):
            return ent
    return "unknown"

def extract_values(text):
    """
    Extract numeric values likely to be metrics.
    Supports:
    - 95.3%
    - 95%
    - 0.953
    - 95.30
    """
    patterns = [
        r"\b\d{1,3}\.\d{1,3}%\b",
        r"\b\d{1,3}%\b",
        r"\b0\.\d{2,4}\b",
        r"\b\d{1,3}\.\d{1,3}\b",
    ]
    vals = []
    for p in patterns:
        vals.extend(re.findall(p, text))
    return list(dict.fromkeys(vals))  # deduplicate while preserving order

cond_patterns = {
    "clean": r"\b(clean|original|orig)\b",
    "JPEG": r"\b(jpeg|jpg)\b",
    "blur": r"\bblur\b",
    "compression": r"\bcompress\w*\b",
    "cross-generator": r"\b(cross-generator|cross generator|cross-domain|transfer)\b",
    "unseen": r"\bunseen\b",
    "robust": r"\brobust\w*\b",
}

def infer_condition(text):
    for cond, pattern in cond_patterns.items():
        if re.search(pattern, text, re.IGNORECASE):
            return cond
    return "unknown"

print(f"Sample table candidate rows loaded: {len(sample_tables)}")

for row in sample_tables:
    # Robustly support different possible field names
    text = get_first_available(
        row,
        ["text", "raw_text", "candidate_text", "content", "line"],
        default=""
    )

    if not isinstance(text, str) or len(text.strip()) == 0:
        continue

    paper_id = get_first_available(row, ["paper_id"], default="unknown")
    evidence_anchor = get_first_available(
        row,
        ["evidence_anchor", "section_id", "candidate_id"],
        default=f"{paper_id}:unknown"
    )

    dataset_guess = find_entity(text, ENTITIES["datasets"])
    metric_guess = find_entity(text, ENTITIES["metrics"])
    values = extract_values(text)
    condition_guess = infer_condition(text)

    # Keep rows that look result-like:
    # either metric present, dataset present, or condition present, plus at least one number.
    looks_result_like = (
        values
        and (
            dataset_guess != "unknown"
            or metric_guess != "unknown"
            or condition_guess != "unknown"
        )
    )

    if looks_result_like:
        for v in values:
            result_results.append({
                "paper_id": paper_id,
                "dataset_guess": dataset_guess,
                "metric_guess": metric_guess,
                "value_guess": v,
                "condition_guess": condition_guess,
                "evidence_anchor": evidence_anchor,
                "raw_text": text[:1000],
            })

output_file = PROBES_DIR / "result_probe_sample.csv"

df_results = pd.DataFrame(
    result_results,
    columns=[
        "paper_id",
        "dataset_guess",
        "metric_guess",
        "value_guess",
        "condition_guess",
        "evidence_anchor",
        "raw_text",
    ]
)

df_results.to_csv(output_file, index=False)

print(f"Extracted {len(result_results)} potential result tuples.")
print(f"Saved: {output_file}")

if len(result_results) > 0:
    from IPython.display import display
    display(df_results.head(10))
else:
    print("No result tuples extracted. This is not fatal for Notebook 02; status may become PROBE_CAUTION.")

Sample table candidate rows loaded: 842
Extracted 313 potential result tuples.
Saved: /content/drive/MyDrive/AIGC/Data/probes/result_probe_sample.csv


,paper_id,dataset_guess,metric_guess,value_guess,condition_guess,evidence_anchor,raw_text
0,P002,unknown,F1,0.99,unknown,P002_C31,"Proposed (AUROC = 0.99, F1 = 0.97, MCC = 0.93)"
1,P002,unknown,F1,0.97,unknown,P002_C31,"Proposed (AUROC = 0.99, F1 = 0.97, MCC = 0.93)"
2,P002,unknown,F1,0.93,unknown,P002_C31,"Proposed (AUROC = 0.99, F1 = 0.97, MCC = 0.93)"
3,P002,unknown,F1,0.99,unknown,P002_C32,"Proposed (AUROC = 0.99, F1 = 0.96, MCC = 0.92)"
4,P002,unknown,F1,0.96,unknown,P002_C32,"Proposed (AUROC = 0.99, F1 = 0.96, MCC = 0.92)"
5,P002,unknown,F1,0.92,unknown,P002_C32,"Proposed (AUROC = 0.99, F1 = 0.96, MCC = 0.92)"
6,P002,unknown,F1,0.99,unknown,P002_C33,"Proposed, Specific (AUROC = 1.00, F1 = 0.99, M..."
7,P002,unknown,F1,0.97,unknown,P002_C33,"Proposed, Specific (AUROC = 1.00, F1 = 0.99, M..."
8,P002,unknown,F1,1.00,unknown,P002_C33,"Proposed, Specific (AUROC = 1.00, F1 = 0.99, M..."
9,P002,unknown,F1,0.99,unknown,P002_C34,"Proposed, Specific (AUROC = 0.99, F1 = 0.97, M..."


## 8. Rule-Based Question Router

In [9]:
def classify_question(q):
    q_lower = q.lower()
    if any(x in q_lower for x in ["highest", "most", "frequent", "distribution"]): return "aggregation"
    if any(x in q_lower for x in ["contradict", "conflict", "disagree"]): return "contradiction"
    if any(x in q_lower for x in ["timeline", "evolution", "before", "after"]): return "temporal"
    if any(x in q_lower for x in ["citation", "cite", "reference"]): return "citation_graph"
    if any(x in q_lower for x in ["how many", "average", "count", "total"]): return "quantitative"
    if "not" in q_lower or "lack" in q_lower: return "negation"
    if "and" in q_lower or "both" in q_lower or "shared" in q_lower: return "multihop"
    return "single_doc"

test_questions = [
    "What datasets are used by P003?",
    "What method does DIRE propose?",
    "Which sample papers mention diffusion-generated images?",
    "Which metrics appear most often in the sample?",
    "Which papers discuss JPEG or compression robustness?",
    "Which datasets are shared across multiple sample papers?",
    "Which paper has the highest citation count among the sample?",
    "Trace the sample timeline from GAN detection to diffusion detection.",
    "Which sample papers lack experimental results sections?",
    "Are any sample papers video-only or audio-only?",
    "What benchmark papers appear in the sample?",
    "Which papers mention CLIP or DINO?"
]

router_report = ["# Question Router Probe Report", ""]
for q in test_questions:
    qtype = classify_question(q)
    router_report.append(f"- **Q**: {q}")
    router_report.append(f"  - **Type**: {qtype}")

output_file = PROBES_DIR / "question_router_probe.md"
with open(output_file, "w") as f:
    f.write("\n".join(router_report))
print(f"Question router report saved: {output_file}")

Question router report saved: /content/drive/MyDrive/AIGC/Data/probes/question_router_probe.md


## 9. Status JSON

In [10]:
if len(sample_sections) == 0:
    status_str = "PROBE_BLOCKED"
elif len(entity_results) > 20 and len(test_questions) == 12:
    status_str = "PROBE_PASS"
elif len(entity_results) > 0:
    status_str = "PROBE_CAUTION"
else:
    status_str = "PROBE_BLOCKED"

status_data = {
    "timestamp": datetime.now().isoformat(),
    "sample_paper_count": len(SAMPLE_PAPER_IDS),
    "sections_loaded": len(sample_sections),
    "table_candidates_loaded": len(sample_tables),
    "entity_rows": len(entity_results),
    "result_rows": len(result_results),
    "router_questions_tested": len(test_questions),
    "final_status": status_str
}

output_file = PROBES_DIR / "notebook02_status.json"
with open(output_file, "w") as f:
    json.dump(status_data, f, indent=4)

print(f"Status file saved: {output_file}")
print(f"Final Status: {status_str}")

Status file saved: /content/drive/MyDrive/AIGC/Data/probes/notebook02_status.json
Final Status: PROBE_PASS


## 10. Recommendation

In [11]:
final_status = status_data["final_status"]
if final_status == "PROBE_PASS":
    print("PROCEED_TO_FULL_EXTRACTION_PIPELINE")
elif final_status == "PROBE_CAUTION":
    print("NEED_RESULT_EXTRACTION_REFINEMENT")
else:
    print("NEED_DATA_SYNC_FIX")

PROCEED_TO_FULL_EXTRACTION_PIPELINE
